In [2]:
## IMPORTS AND SETUP
# Jupyter Notebook setup
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
sys.path.insert(0, "/tf/projet") # Add the project root directory to the Python path (docker hosting)

# Imports
import json
import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from Leyanda_Project.utils.warning_clean import silence_tensorflow_warnings
from wandb.integration.keras import WandbMetricsLogger

# Suppress warnings
silence_tensorflow_warnings()

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU is available: {len(gpus)} device(s) detected")
    except RuntimeError as e:
        print("Error configuring GPU:", str(e))
else:
    print("No GPU available, using CPU")

# Wandb setup
if not os.path.exists("/tf/projet/.env"):
    print("WARNING: No .env file found, please create one with your Wandb API key.")
    exit(1)
else:
    load_dotenv("/tf/projet/.env")
    WANDB_API_KEY = os.getenv("API_KEY")
    wandb_entity = "tom-antoine-cesi"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
TensorFlow warnings suppression is active.
GPU is available: 1 device(s) detected


In [3]:
## PARAMETERS
seed = 123
project_name = "Leyanda"
np.random.seed(seed)
tf.random.set_seed(seed)

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset_livrable_3"  # Path to the raw data folder
images_folder = os.path.join(raw_data_path, "train2017")  # Path to the images folder
annotations_folder = os.path.join(raw_data_path, "annotations")  # Path to the annotations folder

batch_size = 64  # Batch size for dataset loading
max_length = 30  # Maximum caption length
vocab_size_limit = 10000  # Maximum vocabulary size

# Dataset split parameters
train_split = 0.8  # Proportion of the dataset to use for training
val_split = 0.1  # Proportion of the dataset to use for validation
test_split = 0.1  # Proportion of the dataset to use for testing

# Model parameters
embedding_dim = 256  # Dimension of word embeddings
units = 512  # Number of units in LSTM layers

In [4]:
## DATA LOADING AND PREPROCESSING FUNCTIONS
def load_coco_dataset(images_folder, annotations_folder, annotation_file="captions_train2017.json"):
    """
    Load the COCO dataset with images and their captions.

    Parameters:
    ----------
    images_folder : str
        Path to the folder containing images
    annotations_folder : str
        Path to the folder containing annotations
    annotation_file : str, optional
        Name of the annotation file, by default "captions_train2017.json"

    Returns:
    -------
    tuple
        (image_paths, captions) - Lists of image paths and corresponding captions
    """
    print(f"Loading COCO dataset from {images_folder} and {annotations_folder}...")

    annotations_path = os.path.join(annotations_folder, annotation_file)
    with open(annotations_path, 'r') as f:
        annotations_data = json.load(f)

    image_paths = []
    captions = []

    for annotation in annotations_data['annotations']:
        img_id = annotation['image_id']
        img_name = f'{int(img_id):012d}.jpg'
        img_path = os.path.join(images_folder, img_name)

        if os.path.exists(img_path):
            image_paths.append(img_path)
            captions.append(annotation['caption'])

    print(f"Loaded {len(image_paths)} images with captions")
    return image_paths, captions

def create_tokenizer(captions, num_words=10000):
    """
    Create and fit a tokenizer on all captions.

    Parameters:
    ----------
    captions : list
        List of all captions
    num_words : int, optional
        Maximum number of words to keep, by default 10000

    Returns:
    -------
    tuple
        (tokenizer, vocab_size) - Fitted tokenizer and vocabulary size
    """
    print("Creating and fitting tokenizer...")

    tokenizer = Tokenizer(
        num_words=num_words,
        oov_token="<unk>",
        filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ '
    )

    processed_captions = ['<start> ' + caption + ' <end>' for caption in captions]

    tokenizer.fit_on_texts(processed_captions)

    word_index = tokenizer.word_index
    if '<start>' not in word_index:
        word_index['<start>'] = len(word_index) + 1
    if '<end>' not in word_index:
        word_index['<end>'] = len(word_index) + 1

    vocab_size = min(num_words, len(tokenizer.word_index) + 1)
    print(f"Vocabulary size: {vocab_size}")

    return tokenizer, vocab_size


def preprocess_image_path(img_path, target_size=(299, 299)):
    """
    Load and preprocess an image from path.

    Parameters:
    ----------
    img_path : str
        Path to the image
    target_size : tuple, optional
        Target size for resizing, by default (299, 299)

    Returns:
    -------
    tensor
        Preprocessed image tensor
    """
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, target_size)
    img = preprocess_input(img)
    return img


def preprocess_caption(caption, tokenizer, max_length=30):
    """
    Preprocess a caption: add tokens, convert to sequence and pad.

    Parameters:
    ----------
    caption : str
        Caption text
    tokenizer : Tokenizer
        Fitted tokenizer
    max_length : int, optional
        Maximum caption length, by default 30

    Returns:
    -------
    ndarray
        Tokenized and padded caption
    """
    caption = '<start> ' + caption + ' <end>'

    sequence = tokenizer.texts_to_sequences([caption])[0]
    padded_sequence = pad_sequences([sequence], maxlen=max_length, padding='post')[0]

    return padded_sequence


def create_dataset_generator(image_paths, captions, tokenizer, max_length=30, batch_size=32, shuffle=True):
    """
    Create a TensorFlow data generator that yields batches of preprocessed images and captions.

    Parameters:
    ----------
    image_paths : list
        List of image paths
    captions : list
        List of corresponding captions
    tokenizer : Tokenizer
        Fitted tokenizer
    max_length : int, optional
        Maximum caption length, by default 30
    batch_size : int, optional
        Batch size, by default 32
    shuffle : bool, optional
        Whether to shuffle the dataset, by default True

    Returns:
    -------
    tf.data.Dataset
        TensorFlow dataset that yields (image, caption) pairs
    """
    def generator():
        """Generator function that yields preprocessed (image, caption) pairs."""
        indices = list(range(len(image_paths)))
        if shuffle:
            np.random.shuffle(indices)

        for i in indices:
            img_path = image_paths[i]
            caption = captions[i]

            img = preprocess_image_path(img_path, target_size=(299, 299))
            seq = preprocess_caption(caption, tokenizer, max_length)

            yield img, seq

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(299, 299, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
        )
    )

    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset


def split_dataset(image_paths, captions, train_split=0.8, val_split=0.1):
    """
    Split the dataset into training, validation and test sets.

    Parameters:
    ----------
    image_paths : list
        List of image paths
    captions : list
        List of corresponding captions
    train_split : float, optional
        Proportion for training, by default 0.8
    val_split : float, optional
        Proportion for validation, by default 0.1

    Returns:
    -------
    tuple
        (train_img_paths, train_captions, val_img_paths, val_captions, test_img_paths, test_captions)
    """
    print("Splitting dataset into train, validation, and test sets...")

    indices = np.arange(len(image_paths))
    np.random.shuffle(indices)

    train_size = int(train_split * len(image_paths))
    val_size = int(val_split * len(image_paths))

    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size+val_size]
    test_indices = indices[train_size+val_size:]

    train_img_paths = [image_paths[i] for i in train_indices]
    train_captions = [captions[i] for i in train_indices]

    val_img_paths = [image_paths[i] for i in val_indices]
    val_captions = [captions[i] for i in val_indices]

    test_img_paths = [image_paths[i] for i in test_indices]
    test_captions = [captions[i] for i in test_indices]

    print(f"Train set: {len(train_img_paths)} samples")
    print(f"Validation set: {len(val_img_paths)} samples")
    print(f"Test set: {len(test_img_paths)} samples")

    return (train_img_paths, train_captions,
            val_img_paths, val_captions,
            test_img_paths, test_captions)

In [5]:
## DATA PREPARATION WORKFLOW
print(f"Starting data preparation workflow for {project_name}...")

# Step 1: Load COCO dataset
image_paths, captions = load_coco_dataset(
    images_folder=images_folder,
    annotations_folder=annotations_folder
)

# Step 2: Limit dataset size for testing (remove for full training)
max_samples = 10000
if len(image_paths) > max_samples:
    print(f"Limiting dataset to {max_samples} samples for testing")
    random_indices = np.random.choice(len(image_paths), max_samples, replace=False)
    image_paths = [image_paths[i] for i in random_indices]
    captions = [captions[i] for i in random_indices]

# Step 3: Create and fit tokenizer
tokenizer, vocab_size = create_tokenizer(captions, num_words=vocab_size_limit)

# Step 4: Split dataset
(train_img_paths, train_captions,
 val_img_paths, val_captions,
 test_img_paths, test_captions) = split_dataset(
    image_paths,
    captions,
    train_split=train_split,
    val_split=val_split
)

# Step 5: Create TensorFlow datasets
print("Creating TensorFlow datasets...")
train_dataset = create_dataset_generator(
    train_img_paths,
    train_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

val_dataset = create_dataset_generator(
    val_img_paths,
    val_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

test_dataset = create_dataset_generator(
    test_img_paths,
    test_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

print("Data preparation complete!")

for images, captions in train_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Caption batch shape: {captions.shape}")
    break

Starting data preparation workflow for Leyanda...
Loading COCO dataset from /tf/projet/Dataset_livrable_3/train2017 and /tf/projet/Dataset_livrable_3/annotations...
Loaded 591753 images with captions
Limiting dataset to 10000 samples for testing
Creating and fitting tokenizer...
Vocabulary size: 5069
Splitting dataset into train, validation, and test sets...
Train set: 8000 samples
Validation set: 1000 samples
Test set: 1000 samples
Creating TensorFlow datasets...
Data preparation complete!
Image batch shape: (64, 299, 299, 3)
Caption batch shape: (64, 30)
